In [1]:
## Run this cell FIRST

from pyspark.sql import SparkSession
from IPython.core.magic import register_cell_magic


spark = SparkSession.builder \
    .appName("Sedona") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.dynamicAllocation.shuffleTracking.enabled", "true") \
    .config("spark.dynamicAllocation.minExecutors", "0") \
    .config("spark.dynamicAllocation.initialExecutors", "1") \
    .config("spark.dynamicAllocation.maxExecutors", "4") \
    .config("spark.dynamicAllocation.executorIdleTimeout", "10s") \
    .getOrCreate()

@register_cell_magic
def sql(line, cell):

    # Eliminar comentarios SQL de líneas completas
    lines = []
    for line in cell.splitlines():
        if not line.strip().startswith("#"):
            lines.append(line)

    cell = "\n".join(lines)

    # Separar sentencias
    statements = [
        s.strip()
        for s in cell.split(";")
        if s.strip()
    ]

    for statement in statements:
        print("\n==============================")
        print("RUNNING:")
        print(statement)
        print("==============================")

        result = spark.sql(statement)
        result.show(truncate=False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/24 01:30:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/24 01:30:07 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/08/24 01:30:08 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive} is set, falling back to uploading libraries under SPARK_HOME.


In [2]:
%%sql

# Run this cell SECOND


CREATE TABLE IF NOT EXISTS trips
USING DELTA
LOCATION '/spark-warehouse/trips';

CREATE TABLE IF NOT EXISTS boundary
USING DELTA
LOCATION '/spark-warehouse/boundary';

CREATE TABLE IF NOT EXISTS stations
USING DELTA
LOCATION '/spark-warehouse/stations';

CREATE TABLE IF NOT EXISTS nta
USING DELTA
LOCATION '/spark-warehouse/nta';


RUNNING:
CREATE TABLE IF NOT EXISTS trips
USING DELTA
LOCATION '/spark-warehouse/trips'
++
||
++
++


RUNNING:
CREATE TABLE IF NOT EXISTS boundary
USING DELTA
LOCATION '/spark-warehouse/boundary'
++
||
++
++


RUNNING:
CREATE TABLE IF NOT EXISTS stations
USING DELTA
LOCATION '/spark-warehouse/stations'
++
||
++
++


RUNNING:
CREATE TABLE IF NOT EXISTS nta
USING DELTA
LOCATION '/spark-warehouse/nta'
++
||
++
++



In [3]:
print("APP:", spark.sparkContext.applicationId)
print("EXECUTOR MEMORY:", spark.sparkContext.getConf().get("spark.executor.memory"))
print("EXECUTOR CORES:", spark.sparkContext.getConf().get("spark.executor.cores"))
print("EXECUTOR INSTANCES:", spark.sparkContext.getConf().get("spark.executor.instances", "no configurado"))

APP: application_1787528572206_0007
EXECUTOR MEMORY: 4g
EXECUTOR CORES: 1
EXECUTOR INSTANCES: 4


In [4]:
spark.sparkContext.getConf().get("spark.executor.instances", "no configurado")

'4'

In [5]:
print("Dynamic allocation:",
      spark.conf.get("spark.dynamicAllocation.enabled", "false"))

print("Shuffle tracking:",
      spark.conf.get("spark.dynamicAllocation.shuffleTracking.enabled", "false"))

print("Min executors:",
      spark.conf.get("spark.dynamicAllocation.minExecutors", "0"))

print("Initial executors:",
      spark.conf.get("spark.dynamicAllocation.initialExecutors", "1"))

print("Max executors:",
      spark.conf.get("spark.dynamicAllocation.maxExecutors", "0"))

print("Idle timeout:",
      spark.conf.get("spark.dynamicAllocation.executorIdleTimeout", "60s"))

Dynamic allocation: true
Shuffle tracking: true
Min executors: 0
Initial executors: 1
Max executors: 4
Idle timeout: 10s


In [6]:
print(spark.conf.get("spark.dynamicAllocation.enabled"))
print(spark.conf.get("spark.dynamicAllocation.shuffleTracking.enabled"))
print(spark.conf.get("spark.dynamicAllocation.minExecutors"))
print(spark.conf.get("spark.dynamicAllocation.maxExecutors"))

true
true
0
4


In [7]:
%%sql

describe stations;

describe trips;



RUNNING:
describe stations
+---------+---------+-------+
|col_name |data_type|comment|
+---------+---------+-------+
|id       |string   |NULL   |
|name     |string   |NULL   |
|latitude |double   |NULL   |
|longitude|double   |NULL   |
|geometry |geometry |NULL   |
|boroname |string   |NULL   |
|ntacode  |string   |NULL   |
|ntaname  |string   |NULL   |
+---------+---------+-------+


RUNNING:
describe trips
+------------------+---------+-------+
|col_name          |data_type|comment|
+------------------+---------+-------+
|ride_id           |string   |NULL   |
|rideable_type     |string   |NULL   |
|started_at        |timestamp|NULL   |
|ended_at          |timestamp|NULL   |
|start_station_name|string   |NULL   |
|start_station_id  |string   |NULL   |
|end_station_name  |string   |NULL   |
|end_station_id    |string   |NULL   |
|start_lat         |double   |NULL   |
|start_lng         |double   |NULL   |
|end_lat           |double   |NULL   |
|end_lng           |double   |NULL   |
|

In [8]:
%%sql

# Query1: Calculate the monthly distribution of trips that started on Manhattan during 2023

SELECT to_date( date_trunc('month', started_at) ) AS start_month,
repeat('#', CAST((COUNT(*) / 100000) AS INT))
FROM trips t, stations s
WHERE t.start_station_id IS NOT NULL AND
t.start_station_id = s.id AND s.boroname = 'Manhattan' AND
t.started_at BETWEEN '2023-01-01' AND '2023-12-31'
GROUP BY start_month, s.boroname
ORDER BY start_month;



RUNNING:
SELECT to_date( date_trunc('month', started_at) ) AS start_month,
repeat('#', CAST((COUNT(*) / 100000) AS INT))
FROM trips t, stations s
WHERE t.start_station_id IS NOT NULL AND
t.start_station_id = s.id AND s.boroname = 'Manhattan' AND
t.started_at BETWEEN '2023-01-01' AND '2023-12-31'
GROUP BY start_month, s.boroname
ORDER BY start_month


26/08/24 01:31:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 7:=====================================================>   (15 + 1) / 16]

+-----------+-------------------------------------------+
|start_month|repeat(#, CAST((count(1) / 100000) AS INT))|
+-----------+-------------------------------------------+
|2023-01-01 |############                               |
|2023-02-01 |###########                                |
|2023-03-01 |##############                             |
|2023-04-01 |##################                         |
|2023-05-01 |#######################                    |
|2023-06-01 |#######################                    |
|2023-07-01 |########################                   |
|2023-08-01 |##########################                 |
|2023-09-01 |######################                     |
|2023-10-01 |########################                   |
|2023-11-01 |##################                         |
|2023-12-01 |##############                             |
+-----------+-------------------------------------------+



In [9]:
%%sql

# Quey 2: Average trip duration and length per user type. Express the duration in hours and the length in meters.

SELECT member_casual , round(AVG(duration_ms) / (60 * 60 * 1000), 2) AS duration_hours,
AVG(ST_DistanceSphere(origin, target)) AS distance_meters
FROM trips
GROUP BY member_casual;

-- unidad de medida feet. pasar a metros
SELECT member_casual , round(AVG(duration_ms) / (60 * 60 * 1000), 2) AS duration_hours,
AVG(ST_Distance( ST_Transform(origin, 'EPSG:102718' ) , st_transform(target, 'EPSG:102718') )) / 3.28084  AS distance_meters
FROM trips
GROUP BY member_casual;

-- NY corresponde a UTM Zone 18N con unidad de medida en meters
SELECT member_casual , round(AVG(duration_ms) / (60 * 60 * 1000), 2) AS duration_hours,
AVG(ST_Distance( ST_Transform(origin, 'EPSG:32618' ) , st_transform(target, 'EPSG:32618') )) AS distance_meters
FROM trips
GROUP BY member_casual;



RUNNING:
SELECT member_casual , round(AVG(duration_ms) / (60 * 60 * 1000), 2) AS duration_hours,
AVG(ST_DistanceSphere(origin, target)) AS distance_meters
FROM trips
GROUP BY member_casual


+-------------+--------------+------------------+
|member_casual|duration_hours|distance_meters   |
+-------------+--------------+------------------+
|casual       |0.36          |2123.8082035557636|
|member       |0.2           |1866.063533757892 |
+-------------+--------------+------------------+


RUNNING:
-- unidad de medida feet. pasar a metros
SELECT member_casual , round(AVG(duration_ms) / (60 * 60 * 1000), 2) AS duration_hours,
AVG(ST_Distance( ST_Transform(origin, 'EPSG:102718' ) , st_transform(target, 'EPSG:102718') )) / 3.28084  AS distance_meters
FROM trips
GROUP BY member_casual


+-------------+--------------+------------------+
|member_casual|duration_hours|distance_meters   |
+-------------+--------------+------------------+
|casual       |0.36          |2124.4830080837446|
|member       |0.2           |1866.3868848791467|
+-------------+--------------+------------------+


RUNNING:
-- NY corresponde a UTM Zone 18N con unidad de medida en meters
SELECT member_casual , round(AVG(duration_ms) / (60 * 60 * 1000), 2) AS duration_hours,
AVG(ST_Distance( ST_Transform(origin, 'EPSG:32618' ) , st_transform(target, 'EPSG:32618') )) AS distance_meters
FROM trips
GROUP BY member_casual


[Stage 27:====================================================>   (15 + 1) / 16]

+-------------+--------------+------------------+
|member_casual|duration_hours|distance_meters   |
+-------------+--------------+------------------+
|casual       |0.36          |2126.737637018091 |
|member       |0.2           |1866.2232066284805|
+-------------+--------------+------------------+



In [10]:
%%sql

# Quey 3: Number of trips per picking-up borough, year, and month


SELECT boroname, EXTRACT(YEAR FROM t.started_at) AS y,
EXTRACT(MONTH FROM t.started_at) AS m, COUNT(*) AS no_trips
FROM trips t, stations s
WHERE t.start_station_id = s.id and boroname is not null
GROUP BY boroname, y, m
ORDER BY boroname, y, m;



RUNNING:
SELECT boroname, EXTRACT(YEAR FROM t.started_at) AS y,
EXTRACT(MONTH FROM t.started_at) AS m, COUNT(*) AS no_trips
FROM trips t, stations s
WHERE t.start_station_id = s.id and boroname is not null
GROUP BY boroname, y, m
ORDER BY boroname, y, m


[Stage 35:====================================================>   (15 + 1) / 16]

+--------+----+---+--------+
|boroname|y   |m  |no_trips|
+--------+----+---+--------+
|Bronx   |2022|12 |9       |
|Bronx   |2023|1  |42454   |
|Bronx   |2023|2  |41188   |
|Bronx   |2023|3  |56198   |
|Bronx   |2023|4  |98481   |
|Bronx   |2023|5  |145737  |
|Bronx   |2023|6  |141793  |
|Bronx   |2023|7  |143454  |
|Bronx   |2023|8  |147539  |
|Bronx   |2023|9  |111512  |
|Bronx   |2023|10 |103005  |
|Bronx   |2023|11 |82517   |
|Bronx   |2023|12 |69120   |
|Brooklyn|2022|12 |64      |
|Brooklyn|2023|1  |409700  |
|Brooklyn|2023|2  |378585  |
|Brooklyn|2023|3  |474764  |
|Brooklyn|2023|4  |625697  |
|Brooklyn|2023|5  |786741  |
|Brooklyn|2023|6  |797724  |
+--------+----+---+--------+
only showing top 20 rows


In [ ]:
%%sql

# Query 4:  Hourly number of trips that start at a station in Manhattan and end at a station located in a different borough.


SELECT EXTRACT(HOUR FROM t.started_at) AS h, COUNT(*) AS no_trips
FROM trips t, stations s1, stations s2
WHERE t.start_station_id = s1.id AND s1.boroname = 'Manhattan' AND
t.end_station_id = s2.id AND s2.boroname != 'Manhattan'
GROUP BY h
ORDER BY h;

    


RUNNING:
SELECT EXTRACT(HOUR FROM t.started_at) AS h, COUNT(*) AS no_trips
FROM trips t, stations s1, stations s2
WHERE t.start_station_id = s1.id AND s1.boroname = 'Manhattan' AND
t.end_station_id = s2.id AND s2.boroname != 'Manhattan'
GROUP BY h
ORDER BY h


In [ ]:
%%sql

# Query 5: Monthly number of intra- and inter-borough trips per month

WITH TripsBoroughs(y, m, start_borough, end_borough, no_trips) AS (
SELECT EXTRACT(YEAR FROM t.started_at) AS y,
EXTRACT(MONTH FROM t.started_at) AS m,
s1.boroname, s2.boroname, COUNT(*)
FROM trips t, stations s1, stations s2
WHERE t.start_station_id = s1.id AND
t.end_station_id = s2.id
GROUP BY y, m, s1.boroname, s2.boroname ),
IntraTrips(y, m, start_borough, no_intra_trips) AS (
SELECT y, m, start_borough, SUM(no_trips)
FROM TripsBoroughs
WHERE start_borough = end_borough
GROUP BY y, m, start_borough ),
InterTrips(y, m, start_borough, no_inter_trips) AS (
SELECT y, m, start_borough, SUM(no_trips)
FROM TripsBoroughs
WHERE start_borough != end_borough
GROUP BY y, m, start_borough )
SELECT t1.y, t1.m, t1.start_borough, t1.no_intra_trips + t2.no_inter_trips AS no_trips,
t1.no_intra_trips, t2.no_inter_trips
FROM IntraTrips t1, InterTrips t2
WHERE t1.y = t2.y AND t1.m = t2.m AND t1.start_borough = t2.start_borough
ORDER BY t1.y, t1.m, t1.start_borough;



In [ ]:
%%sql

# Query 6: Hourly trips in weekends and weekdays.

WITH Weekday(h, no_weekday) AS (
SELECT EXTRACT(HOUR FROM t.started_at) AS h, COUNT(*)
FROM trips t, stations s
WHERE t.start_station_id = s.id AND
EXTRACT(DOW_ISO FROM t.started_at) IN (1,2,3,4,5)
GROUP BY h ),
Weekend(h, no_weekend) AS (
SELECT EXTRACT(HOUR FROM t.started_at) AS h, COUNT(*)
FROM trips t, stations s
WHERE t.start_station_id = s.id AND
EXTRACT(DOW_ISO FROM t.started_at) IN (6,7)
GROUP BY h )
SELECT w1.h, w1.no_weekday, w2.no_weekend,
w1.no_weekday + w2.no_weekend AS no_trips
FROM Weekday w1, Weekend w2
WHERE w1.h = w2.h
ORDER BY h;



In [ ]:
%%sql

# Query 7: Number of trips per starting station

SELECT s.name AS station_name, s.geometry, COUNT(*) AS no_trips
FROM trips t, stations s
WHERE t.start_station_id = s.id
GROUP BY station_name, geometry
ORDER BY station_name;

In [ ]:
import json
import pydeck as pdk
import pandas as pd

# Visualization Q7: Number of trips per starting station.

df_data = spark.sql("""
SELECT s.name AS station_name, 
    ST_AsGeoJSON(geometry) AS geojson, 
    COUNT(*) AS no_trips
FROM trips t, stations s 
WHERE t.start_station_id = s.id
GROUP BY s.name, geometry
ORDER BY station_name
""").toPandas()


# Generate the quantiles
df_data['quantile_class'] = pd.qcut(df_data['no_trips'], q=8, labels=False, duplicates='drop') + 1



# Define the color palette
color_palette = {
    1: {"rgba":[255, 245, 245, 200], "name": "Off-white / Light gray"},
    2: {"rgba":[250, 222, 211, 200], "name": "Pale orange"},
    3: {"rgba":[255, 180, 203, 200], "name": "Salmon"},
    4: {"rgba":[255, 150, 211, 200], "name": "Coral"},
    5: {"rgba":[245, 120, 103, 200], "name": "Intermediate red"},
    6: {"rgba":[211, 80, 50, 200], "name": "Bright red"},
    7: {"rgba":[180, 40, 30, 200], "name": "Dark red"},
    8: {"rgba":[115, 0, 0, 200], "name": "Deep burgundy / Maroon"}
}

####
print("=== TRIPS INTERVALS ===")
intervals = pd.qcut(df_data['no_trips'], q=8, duplicates='drop')
intervals_mapping = pd.DataFrame({
    'Interval (Trip ranges)': intervals,
    'Quantile Class (Color)': df_data['quantile_class'].map(lambda x: color_palette[x]['name'])
}).drop_duplicates().sort_values(by='Interval (Trip ranges)').reset_index(drop=True)

print(intervals_mapping.to_string(index=False))
print("======================================\n")
####

features_list = [
    {
        "type": "Feature",
        "geometry": json.loads(row["geojson"]),
        "properties": {
            "station_name": str(row["station_name"]),
            "no_trips": int(row["no_trips"]),
            "fill_color": color_palette[int(row["quantile_class"])]["rgba"]
        }
    } for _, row in df_data.iterrows()
]

geojson_data = {"type": "FeatureCollection", "features": features_list}


layer_stations = pdk.Layer(
    "GeoJsonLayer",
    geojson_data,
    stroked=True,
    filled=True,
    get_fill_color="properties.fill_color",
    point_radius_min_pixels=4,
    point_radius_scale=15,
    get_line_color=[100, 100, 100, 255], # Bordes grises de control para los círculos encimados
    line_width_min_pixels=1,
    pickable=True
)


view_state = pdk.ViewState(longitude=-73.96, latitude=40.76, zoom=11)
r = pdk.Deck(
    layers=[layer_stations],
    initial_view_state=view_state,
    map_style=pdk.map_styles.LIGHT,
    tooltip={"html": "<b>Station:</b> {station_name}<br/><b>No_Trips:</b> {no_trips}"}
)

r.show()

In [ ]:
%%sql

# Quey 8: Number of afternoon trips per destination station, starting in Manhattan and ending in another borough

SELECT s2.name AS station_name,
       s2.geometry AS station_geometry,  
       COUNT(*) AS no_trips
FROM trips t, stations s1, stations s2
WHERE t.start_station_id = s1.id AND s1.boroname = 'Manhattan' AND
t.end_station_id = s2.id AND s2.boroname != 'Manhattan' AND
EXTRACT(HOUR FROM t.ended_at) BETWEEN 16 AND 20
GROUP BY s2.name, s2.geometry
ORDER BY s2.name

In [ ]:
import json
import pydeck as pdk
import pandas as pd

# Visualization Q8: Number of afternoon trips per destination station, starting in Manhattan and ending in another borough

df_data = spark.sql("""
SELECT s2.name AS station_name,
       ST_AsGeoJSON(s2.geometry) AS geojson, 
       COUNT(*) AS no_trips
FROM trips t, stations s1, stations s2
WHERE t.start_station_id = s1.id AND s1.boroname = 'Manhattan' AND
t.end_station_id = s2.id AND s2.boroname != 'Manhattan' AND
EXTRACT(HOUR FROM t.ended_at) BETWEEN 16 AND 20
GROUP BY s2.name, s2.geometry
""").toPandas()


# Generate the quantiles
df_data['quantile_class'] = pd.qcut(df_data['no_trips'], q=8, labels=False, duplicates='drop') + 1



# Define the color palette
color_palette = {
    1: {"rgba":[255, 245, 245, 200], "name": "Off-white / Light gray"},
    2: {"rgba":[250, 222, 211, 200], "name": "Pale orange"},
    3: {"rgba":[255, 180, 203, 200], "name": "Salmon"},
    4: {"rgba":[255, 150, 211, 200], "name": "Coral"},
    5: {"rgba":[245, 120, 103, 200], "name": "Intermediate red"},
    6: {"rgba":[211, 80, 50, 200], "name": "Bright red"},
    7: {"rgba":[180, 40, 30, 200], "name": "Dark red"},
    8: {"rgba":[115, 0, 0, 200], "name": "Deep burgundy / Maroon"}
}

####
print("=== TRIPS INTERVALS ===")
intervals = pd.qcut(df_data['no_trips'], q=8, duplicates='drop')
intervals_mapping = pd.DataFrame({
    'Interval (Trip ranges)': intervals,
    'Quantile Class (Color)': df_data['quantile_class'].map(lambda x: color_palette[x]['name'])
}).drop_duplicates().sort_values(by='Interval (Trip ranges)').reset_index(drop=True)

print(intervals_mapping.to_string(index=False))
print("======================================\n")
####

features_list = [
    {
        "type": "Feature",
        "geometry": json.loads(row["geojson"]),
        "properties": {
            "station_name": str(row["station_name"]),
            "no_trips": int(row["no_trips"]),
            "fill_color": color_palette[int(row["quantile_class"])]["rgba"]
        }
    } for _, row in df_data.iterrows()
]

geojson_data = {"type": "FeatureCollection", "features": features_list}


layer_stations = pdk.Layer(
    "GeoJsonLayer",
    geojson_data,
    stroked=True,
    filled=True,
    get_fill_color="properties.fill_color",
    point_radius_min_pixels=4,
    point_radius_scale=15,
    get_line_color=[100, 100, 100, 255], # Bordes grises de control para los círculos encimados
    line_width_min_pixels=1,
    pickable=True
)


view_state = pdk.ViewState(longitude=-73.96, latitude=40.76, zoom=11)
r = pdk.Deck(
    layers=[layer_stations],
    initial_view_state=view_state,
    map_style=pdk.map_styles.LIGHT,
    tooltip={"html": "<b>Station:</b> {station_name}<br/><b>No_Trips:</b> {no_trips}"}
)

r.show()




In [ ]:
%%sql

# Query Q9: Ratio between pick-up and returns in stations located in a radius of 500 m from Columbus circle in Manhattan, between 6 pm and 5 am.

WITH Columbus(geometry) AS (
SELECT ST_Buffer(ST_SetSRID(ST_Point(-73.981676, 40.767984), 4326), 500, true)
),
EndStationsNight(end_station_name, end_station, end_geometry, no_end_trips) AS (
SELECT t.end_station_name, t.end_station_id, s.geometry, COUNT(*)
FROM Columbus c, trips t, stations s
WHERE t.end_station_id = s.id AND ST_Contains(c.geometry, s.geometry) AND
(EXTRACT(HOUR FROM t.started_at) BETWEEN 18 and 23 OR
EXTRACT(HOUR FROM t.started_at) BETWEEN 0 and 5)
GROUP BY t.end_station_name, t.end_station_id, s.geometry),
StartStationsNight(start_station, start_geometry, no_start_trips) AS (
SELECT t.start_station_id, s.geometry, COUNT(*)
FROM Columbus c, trips t, stations s
WHERE t.start_station_id = s.id AND ST_Contains(c.geometry, s.geometry) AND
(EXTRACT(HOUR FROM t.started_at) BETWEEN 18 and 23 OR
EXTRACT(HOUR FROM t.started_at) BETWEEN 0 and 5)
GROUP BY t.start_station_id, s.geometry )
SELECT e.end_station_name AS station, e.end_geometry, s.no_start_trips, e.no_end_trips,
round(e.no_end_trips / s.no_start_trips::numeric, 2) AS ratio
FROM EndStationsNight e, StartStationsNight s
WHERE e.end_station = s.start_station;


In [ ]:
import json
import pydeck as pdk
import pandas as pd

# Visualization Q9: Ratio between pick-up and returns in stations located in a radius of 500 m from Columbus circle in Manhattan, between 6 pm and 5 am.

df_data = spark.sql("""
WITH Columbus(geometry) AS (

SELECT ST_Buffer(ST_SetSRID(ST_Point(-73.981676, 40.767984), 4326), 500,  true)
),

EndStationsNight(end_station_name, end_station, end_geometry, no_end_trips) AS (
SELECT t.end_station_name, t.end_station_id, s.geometry, COUNT(*)
FROM Columbus c, trips t, stations s
WHERE t.end_station_id = s.id AND ST_Contains(c.geometry, s.geometry) AND
(EXTRACT(HOUR FROM t.started_at) BETWEEN 18 and 23 OR
EXTRACT(HOUR FROM t.started_at) BETWEEN 0 and 5)
GROUP BY end_station_name, t.end_station_id, s.geometry),

StartStationsNight(start_station, start_geometry, no_start_trips) AS (
SELECT t.start_station_id, s.geometry, COUNT(*)
FROM Columbus c, trips t, stations s
WHERE t.start_station_id = s.id AND ST_Contains(c.geometry, s.geometry) AND
(EXTRACT(HOUR FROM t.started_at) BETWEEN 18 and 23 OR
EXTRACT(HOUR FROM t.started_at) BETWEEN 0 and 5)
GROUP BY t.start_station_id, s.geometry )

SELECT e.end_station_name AS station, 
ST_AsGeoJSON(e.end_geometry) AS geojson, 
s.no_start_trips, e.no_end_trips,
round(e.no_end_trips / s.no_start_trips::numeric, 2) AS ratio
FROM EndStationsNight e, StartStationsNight s
WHERE e.end_station = s.start_station;

""").toPandas()


features_boundary = [
{
"type": "Feature",
"geometry": json.loads(row["geojson"]),
"properties": {
"station": str(row["station"]),
"ratio": float(row["ratio"])
}
}
for _, row in df_data.iterrows()
]

geojson_boundary = {"type": "FeatureCollection", "features": features_boundary}


layer_station = pdk.Layer(
"GeoJsonLayer",
geojson_boundary,
stroked=True,
filled=True,
get_fill_color=[200, 0, 80, 255],
get_line_color=[200, 0, 80, 255],
get_point_radius=30,  
line_width_min_pixels=3,
pickable=True
)




# LAyer 500 Around Columbus
df_columbus = spark.sql("""
SELECT ST_AsGeoJSON(ST_Buffer(ST_SetSRID(ST_Point(-73.981676, 40.767984), 4326), 500,  true)) as geojson
""").toPandas()

features_columbus = [
{
"type": "Feature",
"geometry": json.loads(row["geojson"])
}
for _, row in df_columbus.iterrows()
]


geojson_columbus = {"type": "FeatureCollection", "features": features_columbus}

layer_columbus = pdk.Layer(
"GeoJsonLayer",
geojson_columbus,
stroked=True,
filled=False,
get_line_color=[200, 220, 80, 255],
line_width_min_pixels=3,
pickable=False
)

# Map centered in New York
view_state = pdk.ViewState(longitude=-73.96, latitude=40.76, zoom=11)

r = pdk.Deck(
layers=[layer_station, layer_columbus],
initial_view_state=view_state,

map_style=pdk.map_styles.LIGHT,
tooltip={"html": "<b>Station:</b> {station}<br/><b>Ratio:</b> {ratio}"}
)


r.show()